# Model 2: Skin Tone Analysis (v2 — UTKFace Dataset)
**Dataset:** UTKFace (Kaggle) — 23,000+ ảnh mặt người  
**Architecture:** EfficientNet-B0 (2-phase training) + K-Means với skin mask  
**Output:** `skin_tone_classifier.pth`, `skin_kmeans.pkl`  
**Target:** Accuracy > 90%

---
### Tại sao dùng UTKFace thay Fitzpatrick17k?
- Fitzpatrick17k: ảnh từ URL bên ngoài, ~50% link chết → khó tải
- UTKFace: 1 file zip ~100MB, tải về 1 lần là xong

### Label mapping (Race → Skin Tone):
| UTKFace Race | Skin Tone |
|---|---|
| 0 = White | Light |
| 2 = Asian | Light |
| 3 = Indian | Medium |
| 4 = Others | Medium |
| 1 = Black | Dark |

### Cách tải dataset:
```bash
# Bước 1: Cài kaggle CLI
pip install kaggle

# Bước 2: Tải API key tại kaggle.com → Account → Create New Token
# Đặt file kaggle.json vào: C:\Users\<tên>\\.kaggle\\kaggle.json

# Bước 3: Tải dataset (chạy trong terminal hoặc cell bên dưới)
kaggle datasets download -d jangedoo/utkface-new
```

In [ ]:
# ============================================================
# CELL 1: Install & Import
# ============================================================
!pip install torch torchvision scikit-learn pandas pillow matplotlib tqdm requests seaborn --quiet

import os, json, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import seaborn as sns
import requests

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ============================================================
# CELL 2: Tải UTKFace từ Kaggle + Giải nén
# ============================================================
import zipfile, shutil

DATA_DIR = "./data/utkface"
IMG_DIR  = f"{DATA_DIR}/images"
os.makedirs(IMG_DIR, exist_ok=True)

ZIP_PATH = f"{DATA_DIR}/utkface-new.zip"

# ── Trỏ Kaggle đọc kaggle.json từ Desktop ──────────────────
KAGGLE_JSON_DIR = r"C:\Users\ACER\OneDrive\Desktop"
os.environ["KAGGLE_CONFIG_DIR"] = KAGGLE_JSON_DIR
print(f"KAGGLE_CONFIG_DIR = {KAGGLE_JSON_DIR}")

# Kiểm tra file tồn tại
kaggle_json = os.path.join(KAGGLE_JSON_DIR, "kaggle.json")
if not os.path.exists(kaggle_json):
    raise FileNotFoundError(f"Khong tim thay: {kaggle_json}\nTai ve tai: kaggle.com → Account → Create New API Token")
print(f"kaggle.json OK: {kaggle_json}")

# ── Tải file zip từ Kaggle ──────────────────────────────────
if not os.path.exists(ZIP_PATH):
    print("Downloading UTKFace từ Kaggle (~100MB)...")
    ret = os.system(f'kaggle datasets download -d jangedoo/utkface-new -p "{DATA_DIR}"')
    if ret != 0:
        print("Loi: kiem tra lai kaggle.json hoac ket noi mang")
    else:
        print("Download xong!")
else:
    print(f"Zip đã có: {ZIP_PATH}")

# ── Giải nén ────────────────────────────────────────────────
img_count = len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')])

if img_count < 1000:
    print(f"Đang giải nén ({img_count} ảnh hiện có)...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        members = [m for m in z.namelist() if m.lower().endswith('.jpg')]
        print(f"Tổng {len(members)} ảnh trong zip")
        for member in tqdm(members, desc="Giải nén"):
            filename = os.path.basename(member)
            if filename:
                target = os.path.join(IMG_DIR, filename)
                if not os.path.exists(target):
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
    img_count = len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')])

print(f"\nTổng ảnh sau giải nén: {img_count}")
print(f"Đường dẫn: {os.path.abspath(IMG_DIR)}")


In [ ]:
# ============================================================
# CELL 3: Parse UTKFace filenames → DataFrame với tone label
# ============================================================
# UTKFace filename: [age]_[gender]_[race]_[datetime].jpg
# race: 0=White, 1=Black, 2=Asian, 3=Indian, 4=Others

# Map race → skin tone (0=Light, 1=Medium, 2=Dark)
RACE_TO_TONE = {
    0: 0,  # White  → Light
    2: 0,  # Asian  → Light
    3: 1,  # Indian → Medium
    4: 1,  # Others → Medium
    1: 2,  # Black  → Dark
}
TONE_LABELS = ['Light', 'Medium', 'Dark']

records = []
skipped = 0

for fname in os.listdir(IMG_DIR):
    if not fname.lower().endswith('.jpg'):
        continue
    parts = fname.replace('.jpg', '').split('_')
    if len(parts) < 3:
        skipped += 1
        continue
    try:
        race = int(parts[2])
        if race not in RACE_TO_TONE:
            skipped += 1
            continue
        tone = RACE_TO_TONE[race]
        records.append({
            'path':        os.path.join(IMG_DIR, fname),
            'race':        race,
            'tone':        tone,
            'fitzpatrick': race,  # Giữ tên cột cho tương thích cell sau
        })
    except:
        skipped += 1

df_local = pd.DataFrame(records)
print(f"Parsed: {len(df_local)} ảnh hợp lệ, bỏ qua: {skipped}")
print("\nPhân bố tone:")
for i, label in enumerate(TONE_LABELS):
    count = (df_local['tone'] == i).sum()
    print(f"  {label:6s} (class {i}): {count:5d} ảnh")

# Cân bằng class — lấy tối đa 3000 ảnh/class để không quá lâu train
MAX_PER_CLASS = 3000
df_local = (
    df_local.groupby('tone')
    .apply(lambda x: x.sample(min(len(x), MAX_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)
print(f"\nSau khi cân bằng (max {MAX_PER_CLASS}/class): {len(df_local)} ảnh")
print(df_local['tone'].value_counts().sort_index())


In [ ]:
# ============================================================
# CELL 4: Define constants + Mixup augmentation helper
# ============================================================
# NOTE: TONE_MAP bên dưới là để tham khảo (Fitzpatrick17k scale 1-6).
# Với UTKFace, tone đã được map đúng từ RACE_TO_TONE ở Cell 3 → KHÔNG ghi đè.

TONE_LABELS = ['Light', 'Medium', 'Dark']
NUM_CLASSES = 3

# Kiểm tra tone distribution (đã được set ở Cell 3)
print("Tone distribution (từ Cell 3):")
print(df_local['tone'].value_counts().sort_index())
print({i: TONE_LABELS[i] for i in range(3)})
assert df_local['tone'].notna().all(), "tone column chứa NaN — kiểm tra lại Cell 3!"


def mixup_data(x, y, alpha=0.4):
    """Mixup augmentation — trộn 2 samples để model tổng quát hóa tốt hơn."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# ============================================================
# CELL 5: Dataset với augmentation cải tiến
# ============================================================
class SkinToneDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = int(row['tone'])
        if self.transform:
            image = self.transform(image)
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08)),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Làm sạch df_local trước khi split ──────────────────────
before = len(df_local)
df_clean = df_local.dropna(subset=['tone', 'path']).copy()
df_clean['tone'] = df_clean['tone'].astype(int)
# Bỏ file không tồn tại hoặc corrupt
df_clean = df_clean[df_clean['path'].apply(os.path.exists)].reset_index(drop=True)
print(f"df_local: {before} → sau khi làm sạch: {len(df_clean)} ảnh")
print("Tone distribution:", df_clean['tone'].value_counts().sort_index().to_dict())

assert len(df_clean) > 0, "df_clean rỗng! Kiểm tra lại Cell 3 và Cell 4."

# 80/20 split, stratified — dùng df_clean (đã loại NaN)
train_df, val_df = train_test_split(
    df_clean, test_size=0.2, stratify=df_clean['tone'], random_state=42
)

train_ds = SkinToneDataset(train_df, train_transform)
val_ds   = SkinToneDataset(val_df,   val_transform)

# num_workers=0 bắt buộc trên Windows
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print("Class distribution (train):", train_df['tone'].value_counts().sort_index().to_dict())

In [ ]:
# ============================================================
# CELL 6: Model setup — EfficientNet-B0
# ============================================================
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

def build_model(num_classes=3, freeze_backbone=True):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes),
    )
    if freeze_backbone:
        # Freeze tất cả trừ classifier
        for name, param in model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False
        print("Backbone FROZEN — chỉ train classifier head")
    else:
        print("Toàn bộ model UNFROZEN")
    return model


# Class weights
class_counts = train_df['tone'].value_counts().sort_index()
total   = class_counts.sum()
weights = torch.tensor(
    [total / (NUM_CLASSES * c) for c in class_counts.values],
    dtype=torch.float32
).to(DEVICE)
print("Class weights:", weights.cpu().numpy().round(3))

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

# Phase 1: Freeze backbone
model = build_model(num_classes=NUM_CLASSES, freeze_backbone=True).to(DEVICE)

# Phase 1 optimizer — lr cao hơn vì chỉ train head
optimizer_p1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-3
)
scheduler_p1 = CosineAnnealingLR(optimizer_p1, T_max=5, eta_min=1e-5)

print(f"\nEfficientNet-B0 ready — {NUM_CLASSES} classes: {TONE_LABELS}")

In [ ]:
# ============================================================
# CELL 7: Phase 1 — Train chỉ classifier head (5 epochs)
# Mục đích: Khởi tạo head tốt trước khi unfreeze backbone
# ============================================================
PHASE1_EPOCHS = 5
history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

print("=" * 60)
print("PHASE 1: Training classifier head only")
print("=" * 60)

for epoch in range(1, PHASE1_EPOCHS + 1):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f'P1 Epoch {epoch:02d}', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        # Mixup
        imgs_m, labels_a, labels_b, lam = mixup_data(imgs, labels, alpha=0.4)
        optimizer_p1.zero_grad()
        out  = model(imgs_m)
        loss = mixup_criterion(criterion, out, labels_a, labels_b, lam)
        loss.backward()
        optimizer_p1.step()
        t_loss    += loss.item() * imgs.size(0)
        t_correct += out.argmax(1).eq(labels).sum().item()  # Report accuracy on unmixed
        t_total   += imgs.size(0)

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            v_loss    += loss.item() * imgs.size(0)
            v_correct += out.argmax(1).eq(labels).sum().item()
            v_total   += imgs.size(0)

    scheduler_p1.step()
    train_acc = t_correct / t_total
    val_acc   = v_correct / v_total
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(t_loss / t_total)
    history['val_loss'].append(v_loss / v_total)
    print(f"[P1 {epoch:02d}/{PHASE1_EPOCHS}] Train: {train_acc:.4f}  Val: {val_acc:.4f}  LR: {scheduler_p1.get_last_lr()[0]:.2e}")

In [ ]:
# ============================================================
# CELL 8: Phase 2 — Unfreeze toàn bộ, fine-tune với lr nhỏ
# ============================================================
# Unfreeze toàn bộ backbone
for param in model.parameters():
    param.requires_grad = True
print("Backbone UNFROZEN")

PHASE2_EPOCHS = 25
TOTAL_EPOCHS  = PHASE1_EPOCHS + PHASE2_EPOCHS

# Phase 2 optimizer — lr nhỏ hơn nhiều để không phá pretrained features
optimizer_p2 = torch.optim.AdamW(
    [
        {'params': model.features.parameters(), 'lr': 3e-5},    # backbone lr rất nhỏ
        {'params': model.classifier.parameters(), 'lr': 2e-4},  # head lr lớn hơn
    ],
    weight_decay=1e-3
)

warmup  = LinearLR(optimizer_p2, start_factor=0.1, end_factor=1.0, total_iters=3)
cosine  = CosineAnnealingLR(optimizer_p2, T_max=PHASE2_EPOCHS - 3, eta_min=1e-7)
scheduler_p2 = SequentialLR(optimizer_p2, schedulers=[warmup, cosine], milestones=[3])

best_val_acc   = max(history['val_acc'])  # Tiếp tục từ phase 1
patience       = 8
patience_count = 0

print("=" * 60)
print("PHASE 2: Fine-tuning full model")
print("=" * 60)

for epoch in range(1, PHASE2_EPOCHS + 1):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f'P2 Epoch {epoch:02d}', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs_m, labels_a, labels_b, lam = mixup_data(imgs, labels, alpha=0.3)
        optimizer_p2.zero_grad()
        out  = model(imgs_m)
        loss = mixup_criterion(criterion, out, labels_a, labels_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_p2.step()
        t_loss    += loss.item() * imgs.size(0)
        t_correct += out.argmax(1).eq(labels).sum().item()
        t_total   += imgs.size(0)

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            v_loss    += loss.item() * imgs.size(0)
            v_correct += out.argmax(1).eq(labels).sum().item()
            v_total   += imgs.size(0)

    scheduler_p2.step()
    train_acc = t_correct / t_total
    val_acc   = v_correct / v_total
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(t_loss / t_total)
    history['val_loss'].append(v_loss / v_total)

    backbone_lr = optimizer_p2.param_groups[0]['lr']
    print(f"[P2 {epoch:02d}/{PHASE2_EPOCHS}] Train: {train_acc:.4f}  Val: {val_acc:.4f}  backbone_lr: {backbone_lr:.2e}")

    if val_acc > best_val_acc:
        best_val_acc   = val_acc
        patience_count = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'tone_labels':  TONE_LABELS,
            'race_to_tone': RACE_TO_TONE,
            'num_classes':  NUM_CLASSES,
            'val_acc':      val_acc,
            'epoch':        PHASE1_EPOCHS + epoch,
        }, 'skin_tone_classifier.pth')
        print(f"  --> Saved best (val_acc={val_acc:.4f})")
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\nBest Val Accuracy: {best_val_acc*100:.1f}%")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history['train_acc'], label='Train')
ax1.plot(history['val_acc'],   label='Val')
ax1.axvline(x=PHASE1_EPOCHS - 0.5, color='red', linestyle='--', label='Unfreeze point')
ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.plot(history['train_loss'], label='Train')
ax2.plot(history['val_loss'],   label='Val')
ax2.axvline(x=PHASE1_EPOCHS - 0.5, color='red', linestyle='--', label='Unfreeze point')
ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_history.png', dpi=100)
plt.show()

In [ ]:
# ============================================================
# CELL 9: K-Means CẢI TIẾN — Skin mask + ITA mapping
# ============================================================
# Fix 2 vấn đề của v1:
#   1. Chỉ lấy pixel vùng da (skin mask) — loại background/quần áo
#   2. Sắp xếp cluster theo độ sáng (L channel) → mapping đúng Light/Medium/Dark

SEASON_MAP = {
    'Light':  'Summer',
    'Medium': 'Autumn',
    'Dark':   'Winter',
}

COLOR_SUGGESTIONS = {
    'Summer': ['pastel pink', 'lavender', 'soft blue', 'mint', 'rose'],
    'Autumn': ['burnt orange', 'olive green', 'mustard', 'terracotta', 'rust'],
    'Winter': ['black', 'white', 'navy', 'royal blue', 'bright red', 'emerald'],
}


def extract_skin_pixels(image_path, resize=(80, 80)):
    """
    Trích xuất pixel vùng da dựa trên HSV skin mask.
    Trả về mảng (N, 3) RGB của các pixel da, hoặc None nếu không đủ pixel.
    """
    try:
        img = Image.open(image_path).convert('RGB').resize(resize)
        arr = np.array(img, dtype=np.float32)

        # Skin detection trong không gian HSV
        r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
        max_c = arr.max(axis=2)
        min_c = arr.min(axis=2)
        diff  = max_c - min_c + 1e-6

        # H (Hue) tính approximate
        h = np.where(max_c == r, (g - b) / diff % 6,
            np.where(max_c == g, (b - r) / diff + 2,
                     (r - g) / diff + 4)) * 60
        s = np.where(max_c > 0, diff / max_c, 0)  # Saturation
        v = max_c / 255.0  # Value

        # Skin mask: hue trong khoảng da (0-25 độ), saturation và value hợp lý
        # Đủ rộng để bắt được da từ rất trắng đến rất tối
        skin_mask = (
            ((h >= 0) & (h <= 25)) |
            ((h >= 335) & (h <= 360))
        ) & (s > 0.1) & (s < 0.9) & (v > 0.15)

        skin_pixels = arr[skin_mask].reshape(-1, 3)
        if len(skin_pixels) < 20:  # Không đủ pixel da
            # Fallback: dùng trung tâm ảnh (thường là mặt)
            h_, w_ = resize
            center = arr[h_//4:3*h_//4, w_//4:3*w_//4].reshape(-1, 3)
            return center
        return skin_pixels
    except:
        return None


def compute_ita(r, g, b):
    """Individual Typology Angle — metric chuẩn để đo tone da trong LAB."""
    # Chuyển RGB → LAB
    arr = np.array([[[r, g, b]]], dtype=np.uint8)
    from PIL import Image as PILImage
    import colorsys
    # Simple approximation: sáng hơn → ITA dương, tối hơn → ITA âm
    # L ≈ 0.299R + 0.587G + 0.114B (luminance)
    L = 0.299 * r + 0.587 * g + 0.114 * b
    b_channel = b - (r + g) / 2  # Rough b* in LAB
    ita = np.degrees(np.arctan2(L - 50, b_channel + 1e-6))
    return ita


# Thu thập skin pixels từ ảnh có label
print("Extracting skin pixels...")
skin_features = []  # (mean_R, mean_G, mean_B)
skin_labels_arr = []

sample_size = min(500, len(df_local))  # Tăng từ 300 → 500
for _, row in tqdm(
    df_local.sample(sample_size, random_state=42).iterrows(),
    total=sample_size, desc="Skin pixels"
):
    pixels = extract_skin_pixels(row['path'])
    if pixels is not None and len(pixels) > 0:
        mean_rgb = pixels.mean(axis=0)  # (R_mean, G_mean, B_mean)
        skin_features.append(mean_rgb)
        skin_labels_arr.append(int(row['tone']))

skin_features   = np.array(skin_features)
skin_labels_arr = np.array(skin_labels_arr)
print(f"Collected {len(skin_features)} samples with skin pixels")

# Train KMeans
kmeans_skin = KMeans(n_clusters=3, n_init=30, random_state=42, max_iter=500)
kmeans_skin.fit(skin_features)

# === FIX CHÍNH: Sắp xếp cluster theo độ sáng → map đúng Light/Medium/Dark ===
centers = kmeans_skin.cluster_centers_  # (3, 3) — 3 clusters, mỗi cluster là [R, G, B]
# Tính luminance của mỗi cluster center
luminance = np.array([
    0.299 * c[0] + 0.587 * c[1] + 0.114 * c[2]
    for c in centers
])
# Sắp xếp từ sáng nhất (Light) → tối nhất (Dark)
sorted_cluster_idx = np.argsort(-luminance)  # Descending brightness
# cluster_to_tone: cluster_id → tone_class (0=Light, 1=Medium, 2=Dark)
cluster_to_tone = {int(sorted_cluster_idx[i]): i for i in range(3)}

print("\nCluster centers (RGB):")
for ci in range(3):
    tone_idx  = cluster_to_tone[ci]
    tone_name = TONE_LABELS[tone_idx]
    lum       = luminance[ci]
    print(f"  Cluster {ci} → {tone_name:6s} | RGB={centers[ci].astype(int)} | Luminance={lum:.1f}")

# Đánh giá accuracy của KMeans fallback trên tập đã có label
km_preds  = kmeans_skin.predict(skin_features)
km_tones  = np.array([cluster_to_tone[int(p)] for p in km_preds])
km_acc    = (km_tones == skin_labels_arr).mean()
print(f"\nKMeans fallback accuracy (skin pixels): {km_acc*100:.1f}%")
print("(so sánh với random baseline ~33%)")

# Lưu model với cluster_to_tone mapping
with open('skin_kmeans.pkl', 'wb') as f:
    pickle.dump({
        'kmeans':          kmeans_skin,
        'tone_labels':     TONE_LABELS,
        'season_map':      SEASON_MAP,
        'color_suggestions': COLOR_SUGGESTIONS,
        'cluster_to_tone': cluster_to_tone,   # FIX: mapping đúng
        'cluster_centers': centers.tolist(),
        'luminance_order': sorted_cluster_idx.tolist(),
    }, f)

print("\nskin_kmeans.pkl saved (v2 — với cluster_to_tone mapping)")

In [ ]:
# ============================================================
# CELL 10: Evaluation với TTA (Test Time Augmentation)
# ============================================================
# Load best checkpoint
checkpoint = torch.load('skin_tone_classifier.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model (epoch {checkpoint['epoch']}, val_acc={checkpoint['val_acc']:.4f})")

# TTA transforms — 5 views
tta_transforms = [
    transforms.Compose([  # Original
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([  # Horizontal flip
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([  # Slight brightness up
        transforms.Resize((224, 224)),
        transforms.ColorJitter(brightness=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([  # Slight brightness down
        transforms.Resize((224, 224)),
        transforms.ColorJitter(brightness=(0.85, 0.85)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([  # Center crop
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
]


def predict_with_tta(model, image_path, tta_transforms, device):
    """Average predictions over TTA transforms."""
    img = Image.open(image_path).convert('RGB')
    all_probs = []
    with torch.no_grad():
        for tfm in tta_transforms:
            tensor = tfm(img).unsqueeze(0).to(device)
            out    = model(tensor)
            probs  = F.softmax(out, dim=1)[0]
            all_probs.append(probs.cpu().numpy())
    avg_probs = np.mean(all_probs, axis=0)
    return int(np.argmax(avg_probs)), avg_probs


# Evaluate on val set
print("\nEvaluating without TTA...")
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out   = model(imgs.to(DEVICE))
        preds = out.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

acc_no_tta = (np.array(all_preds) == np.array(all_labels)).mean()
print(f"Val Accuracy (no TTA): {acc_no_tta*100:.2f}%")
print(classification_report(all_labels, all_preds, target_names=TONE_LABELS))

# TTA evaluation (chậm hơn, chỉ dùng subset)
print("\nEvaluating with TTA (subset 200 ảnh)...")
tta_preds, tta_labels = [], []
val_subset = val_df.sample(min(200, len(val_df)), random_state=42)
for _, row in tqdm(val_subset.iterrows(), total=len(val_subset)):
    pred, _ = predict_with_tta(model, row['path'], tta_transforms, DEVICE)
    tta_preds.append(pred)
    tta_labels.append(int(row['tone']))

acc_tta = (np.array(tta_preds) == np.array(tta_labels)).mean()
print(f"Val Accuracy (with TTA): {acc_tta*100:.2f}%")
print(f"TTA improvement: +{(acc_tta - acc_no_tta)*100:.2f}%")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=TONE_LABELS,
            yticklabels=TONE_LABELS, cmap='OrRd')
plt.title(f'Confusion Matrix — Val Acc: {acc_no_tta*100:.1f}%')
plt.tight_layout()
plt.savefig('confusion_matrix_skin.png', dpi=100)
plt.show()

In [ ]:
# ============================================================
# CELL 11: Inference function (production-ready)
# Tương thích 100% với backend ai_service.py
# ============================================================

def analyze_skin_tone_v2(image_path, cnn_model, kmeans_data, device, use_tta=True):
    """
    Full inference pipeline với TTA.
    Returns: dict với tone_class, confidence, season, recommended_colors.
    """
    # CNN prediction
    if use_tta:
        pred_idx, probs = predict_with_tta(cnn_model, image_path, tta_transforms, device)
    else:
        transform = val_transform
        img = Image.open(image_path).convert('RGB')
        tensor = transform(img).unsqueeze(0).to(device)
        cnn_model.eval()
        with torch.no_grad():
            out    = cnn_model(tensor)
            probs  = F.softmax(out, dim=1)[0].cpu().numpy()
            pred_idx = int(np.argmax(probs))

    tone_label = kmeans_data['tone_labels'][pred_idx]
    season     = kmeans_data['season_map'][tone_label]
    colors     = kmeans_data['color_suggestions'][season]

    return {
        'tone_class':        tone_label,
        'confidence':        float(probs[pred_idx]),
        'all_probs':         {kmeans_data['tone_labels'][i]: float(probs[i]) for i in range(3)},
        'season':            season,
        'recommended_colors': colors,
    }


# Load saved models
checkpoint   = torch.load('skin_tone_classifier.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

with open('skin_kmeans.pkl', 'rb') as f:
    kmeans_data = pickle.load(f)

# Verify cluster_to_tone exists
assert 'cluster_to_tone' in kmeans_data, "skin_kmeans.pkl thiếu cluster_to_tone!"
print("cluster_to_tone mapping:", kmeans_data['cluster_to_tone'])

# Test trên vài ảnh
for i, row in val_df.sample(5, random_state=7).iterrows():
    result = analyze_skin_tone_v2(row['path'], model, kmeans_data, DEVICE, use_tta=True)
    gt     = TONE_LABELS[int(row['tone'])]
    status = '✓' if result['tone_class'] == gt else '✗'
    print(f"{status} GT={gt:6s} | Pred={result['tone_class']:6s} | conf={result['confidence']:.2f} | {result['season']}")

In [ ]:
# ============================================================
# CELL 12: Update ai_service.py — Fix KMeans fallback
# ============================================================
# Đoạn code này show patch cần áp dụng cho backend
# (Đã được apply trong ai_service.py hiện tại)

patch_note = """
PATCH cho backend/app/services/ai_service.py — KMeans fallback (dòng ~225):

TRƯỚC (v1 — sai):
    tone_label = tone_labels[cluster % len(tone_labels)]

SAU (v2 — đúng):
    cluster_to_tone = kmeans_data.get('cluster_to_tone', {0: 0, 1: 1, 2: 2})
    tone_class = cluster_to_tone.get(int(cluster), 1)
    tone_label = tone_labels[tone_class]

Lý do: KMeans cluster index KHÔNG tương ứng với tone class theo thứ tự.
Cần dùng cluster_to_tone map được tính từ luminance khi training.
"""
print(patch_note)

# Copy models sang backend
import shutil
BACKEND_MODELS = '../backend/app/ai_models/'
if os.path.exists(BACKEND_MODELS):
    shutil.copy('skin_tone_classifier.pth', BACKEND_MODELS)
    shutil.copy('skin_kmeans.pkl',          BACKEND_MODELS)
    print(f"Models copied to {BACKEND_MODELS}")
else:
    print(f"Backend path not found: {BACKEND_MODELS} — copy thủ công nếu cần")